# 07 — Filtering DataFrames: Boolean Indexing, Deduplication & Memory Optimization
> **Interview Prep & Technical Mastery Guide**
> 
> *A comprehensive, battle-tested reference for Python Data Science, Machine Learning, and Analytics Interviews.*

---

## 📌 Executive Summary & Interview Expectations
Filtering data is the most common operation in data manipulation. In technical interviews, interviewers test your ability to filter datasets **without introducing silent data bugs**, **without memory bloat**, and **without falling into common Python/Pandas logic traps**.

### Core Competencies Tested in this Module:
1. **Memory Optimization Traps**: Why `astype(bool)` silently corrupts missing data into `True`, and how modern `astype("boolean")` solves it.
2. **Boolean Indexing Mastery**: Bitwise operators (`&`, `|`, `~`), operator precedence rules, and parentheses requirements.
3. **Range & Membership Filtering**: `.between()` with date/numeric intervals and `.isin()` vs cascading `|` clauses.
4. **Missing Value Filtering**: The `NaN != NaN` IEEE 754 float rule, `.isna()`, `.notna()`, and `.dropna(subset=..., thresh=...)`.
5. **Deduplication Mechanics**: The difference between `keep='first'`, `keep='last'`, and `keep=False` in `.duplicated()`.
6. **Interview Corner**: The `bool(NaN)` corruption bug, inverted boolean masking with `~`, and team-relative filtering drills.

## 1. Environment Setup & Data Ingestion
We load both `employees.csv` and `netflix.csv` with automated remote fallbacks.

In [1]:
import os
import numpy as np
import pandas as pd

# Load employees dataset
emp_path = "employees.csv"
if not os.path.exists(emp_path):
    emp_path = "https://raw.githubusercontent.com/paskhaver/pandas-in-action/master/chapter_05_filtering_a_dataframe/employees.csv"

employees = pd.read_csv(emp_path, parse_dates=["Start Date"])
print("Employees dataset loaded. Shape:", employees.shape)
employees.head()

Employees dataset loaded. Shape: (1001, 6)


/var/folders/54/5j97z6452x19yddj6yv2l7j00000gn/T/ipykernel_26507/3175463766.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  employees = pd.read_csv(emp_path, parse_dates=["Start Date"])


,First Name,Gender,Start Date,Salary,Mgmt,Team
0,Douglas,Male,1993-08-06,NaN,True,Marketing
1,Thomas,Male,1996-03-31,61933.0,True,NaN
2,Maria,Female,NaT,130590.0,False,Finance
3,Jerry,NaN,2005-03-04,138705.0,True,Finance
4,Larry,Male,1998-01-24,101004.0,True,IT


## 2. Memory Optimization & The Silent `astype(bool)` Trap

### ⚠️ Top Interview Trap: `astype(bool)` vs `astype("boolean")`
- In Python, `bool(np.nan)` evaluates to **`True`** because `np.nan` is a non-zero float!
- If you cast a column with missing values using `df['Mgmt'].astype(bool)`, **all `NaN` values are converted into `True`**! This silently turns non-managers or unknown roles into managers.
- **The Modern Fix**: Use Pandas' nullable **`"boolean"`** dtype (backed by `pd.NA`). It preserves three states: `True`, `False`, and `<NA>`!

In [2]:
# Demonstration of the silent boolean corruption trap
sample_mgmt = employees["Mgmt"].copy()
print("Original missing count in Mgmt:", sample_mgmt.isna().sum())

# ❌ BAD: astype(bool) turns NaNs into True!
corrupted_bool = sample_mgmt.astype(bool)
print("Missing count after astype(bool):", corrupted_bool.isna().sum(), "(NaNs became True!)")

# ✅ GOOD: astype("boolean") preserves True, False, and <NA>
safe_bool = sample_mgmt.astype("boolean")
print("Missing count after astype('boolean'):", safe_bool.isna().sum(), "(Preserved!)")

Original missing count in Mgmt: 68
Missing count after astype(bool): 0 (NaNs became True!)
Missing count after astype('boolean'): 68 (Preserved!)


In [3]:
# Apply memory optimization safely across employees
employees["Mgmt"] = employees["Mgmt"].astype("boolean")
employees["Salary"] = employees["Salary"].fillna(0).astype("int64")
employees["Gender"] = employees["Gender"].astype("category")
employees["Team"] = employees["Team"].astype("category")

employees.info()

<class 'pandas.DataFrame'>
RangeIndex: 1001 entries, 0 to 1000
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   First Name  933 non-null    str           
 1   Gender      854 non-null    category      
 2   Start Date  999 non-null    datetime64[us]
 3   Salary      1001 non-null   int64         
 4   Mgmt        933 non-null    boolean       
 5   Team        957 non-null    category      
dtypes: boolean(1), category(2), datetime64[us](1), int64(1), str(1)
memory usage: 27.6 KB


## 3. Filtering by Single & Multiple Conditions

### ⚠️ Top Interview Rule: Parentheses & Bitwise Operators
- Use bitwise `&` (AND), `|` (OR), and `~` (NOT).
- Python's comparison operators (`==`, `<`, `>`) have **lower precedence** than `&` and `|`.
- Therefore, **every condition MUST be wrapped in parentheses**:
  ```python
  df[(df['A'] == 1) & (df['B'] == 2)]
  ```

In [4]:
# 1. Single condition: Employees in Finance
in_finance = employees["Team"] == "Finance"
employees[in_finance].head()

,First Name,Gender,Start Date,Salary,Mgmt,Team
2,Maria,Female,NaT,130590,False,Finance
3,Jerry,NaN,2005-03-04,138705,True,Finance
7,NaN,Female,2015-07-20,45906,<NA>,Finance
14,Kimberly,Female,1999-01-14,41426,True,Finance
46,Bruce,Male,2009-11-28,114796,False,Finance


In [5]:
# 2. Logical AND: Female employees in Business Development
is_female = employees["Gender"] == "Female"
in_biz_dev = employees["Team"] == "Business Dev"

employees[is_female & in_biz_dev].head()

,First Name,Gender,Start Date,Salary,Mgmt,Team
9,Frances,Female,2002-08-08,139852,True,Business Dev
33,Jean,Female,1993-12-18,119082,False,Business Dev
36,Rachel,Female,2009-02-16,142032,False,Business Dev
38,Stephanie,Female,1986-09-13,36844,True,Business Dev
61,Denise,Female,2001-11-06,106862,False,Business Dev


In [6]:
# 3. Logical OR: Senior Management OR High Salary (>= $100,000)
is_mgmt = employees["Mgmt"] == True
high_salary = employees["Salary"] >= 100_000

employees[is_mgmt | high_salary].head()

,First Name,Gender,Start Date,Salary,Mgmt,Team
0,Douglas,Male,1993-08-06,0,True,Marketing
1,Thomas,Male,1996-03-31,61933,True,NaN
2,Maria,Female,NaT,130590,False,Finance
3,Jerry,NaN,2005-03-04,138705,True,Finance
4,Larry,Male,1998-01-24,101004,True,IT


## 4. Range (`.between()`) and Membership (`.isin()`) Filtering

> 💡 **Interview Tip — `.isin()` vs Chained `|`**:
> Instead of writing `(df['Team'] == 'Sales') | (df['Team'] == 'Marketing') | (df['Team'] == 'Finance')`,
> always use `df['Team'].isin(['Sales', 'Marketing', 'Finance'])`.
> It is faster, more concise, and handles empty sets gracefully.

In [7]:
# Multi-category filtering with .isin()
target_teams = employees["Team"].isin(["Sales", "Marketing", "Finance"])
employees[target_teams].head()

,First Name,Gender,Start Date,Salary,Mgmt,Team
0,Douglas,Male,1993-08-06,0,True,Marketing
2,Maria,Female,NaT,130590,False,Finance
3,Jerry,NaN,2005-03-04,138705,True,Finance
7,NaN,Female,2015-07-20,45906,<NA>,Finance
13,Gary,Male,2008-01-27,109831,False,Sales


In [8]:
# Range filtering with .between() (inclusive of boundaries by default)
mid_range_salary = employees["Salary"].between(80_000, 90_000)
employees[mid_range_salary].head()

,First Name,Gender,Start Date,Salary,Mgmt,Team
19,Donna,Female,2010-07-22,81014,False,Product
31,Joyce,NaN,2005-02-20,88657,False,Product
35,Theresa,Female,2006-10-10,85182,False,Sales
45,Roger,Male,1980-04-17,88010,True,Sales
54,Sara,Female,2007-08-15,83677,False,Engineering


## 5. Identifying & Handling Missing Values (`isna`, `notna`, `dropna`)

### ⚠️ Top Interview Trap: `df['col'] == np.nan`
Never write `df[df['col'] == np.nan]`!
Under IEEE 754 floating point standards, `np.nan == np.nan` evaluates to **`False`**.
Writing `df['col'] == np.nan` will return an empty DataFrame every time! Always use `df['col'].isna()` or `df['col'].notna()`.

In [9]:
# Check why == np.nan fails
print("np.nan == np.nan evaluates to:", np.nan == np.nan)

# Finding employees with missing team information
no_team = employees["Team"].isna()
print("Employees with missing Team:", no_team.sum())
employees[no_team].head()

np.nan == np.nan evaluates to: False
Employees with missing Team: 44


,First Name,Gender,Start Date,Salary,Mgmt,Team
1,Thomas,Male,1996-03-31,61933,True,NaN
10,Louise,Female,1980-08-12,63241,True,NaN
23,NaN,Male,2012-06-14,125792,<NA>,NaN
32,NaN,Male,1998-08-21,122340,<NA>,NaN
91,James,NaN,2005-01-26,128771,False,NaN


In [10]:
# Dropping rows where critical information is missing
# subset parameter ensures we only drop if specific columns are null
clean_employees = employees.dropna(subset=["Team", "First Name"])
print(f"Original rows: {len(employees)} | Rows after dropna: {len(clean_employees)}")

Original rows: 1001 | Rows after dropna: 899


## 6. Deduplication: `.duplicated()` & `.drop_duplicates()`

### ⚠️ Top Interview Distinction: `keep='first'` vs `'last'` vs `False`
| `keep` Option | Meaning in `.duplicated()` | Meaning in `.drop_duplicates()` |
| :--- | :--- | :--- |
| `'first'` (default) | Marks all occurrences **after the first** as duplicate (`True`) | Keeps the 1st occurrence, drops subsequent ones |
| `'last'` | Marks all occurrences **before the last** as duplicate (`True`) | Keeps the last occurrence, drops preceding ones |
| `False` | Marks **ALL occurrences of duplicated values** as `True` | **Drops ALL occurrences**, leaving only strictly unique records! |

In [11]:
# Find duplicate first names (keep='first' flags 2nd+ appearances)
dup_first = employees["First Name"].duplicated(keep="first")

# Find names that appear strictly ONCE (unique across the entire dataset)
all_dups = employees["First Name"].duplicated(keep=False)
strictly_unique_names = employees[~all_dups]

print("Total duplicate name occurrences:", dup_first.sum())
print("Employees with completely unique first names:", len(strictly_unique_names))
strictly_unique_names.head(3)

Total duplicate name occurrences: 800
Employees with completely unique first names: 9


,First Name,Gender,Start Date,Salary,Mgmt,Team
5,Dennis,Male,1987-04-18,115163,False,Legal
8,Angela,Female,2005-11-22,95570,True,Engineering
33,Jean,Female,1993-12-18,119082,False,Business Dev


In [12]:
# Dropping duplicate team entries to get one representative employee per team
unique_team_reps = employees.drop_duplicates(subset=["Team"], keep="first")
unique_team_reps[["Team", "First Name", "Salary"]]

,Team,First Name,Salary
0,Marketing,Douglas,0
1,NaN,Thomas,61933
2,Finance,Maria,130590
4,IT,Larry,101004
5,Legal,Dennis,115163
6,Product,Ruby,65476
8,Engineering,Angela,95570
9,Business Dev,Frances,139852
12,HR,Brandon,112807
13,Sales,Gary,109831


## 7. Practical Exercise: Netflix Catalog Filtering

In [13]:
# Load Netflix catalog
netflix_path = "netflix.csv"
if not os.path.exists(netflix_path):
    netflix_path = "https://raw.githubusercontent.com/paskhaver/pandas-in-action/master/chapter_05_filtering_a_dataframe/netflix.csv"

netflix = pd.read_csv(netflix_path, parse_dates=["date_added"])
netflix["type"] = netflix["type"].astype("category")
netflix.head()

/var/folders/54/5j97z6452x19yddj6yv2l7j00000gn/T/ipykernel_26507/3586167868.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  netflix = pd.read_csv(netflix_path, parse_dates=["date_added"])


,title,director,date_added,type
0,Alias Grace,NaN,2017-11-03,TV Show
1,A Patch of Fog,Michael Lennox,2017-04-15,Movie
2,Lunatics,NaN,2019-04-19,TV Show
3,Uriyadi 2,Vijay Kumar,2019-08-02,Movie
4,Shrek the Musical,Jason Moore,2013-12-29,Movie


In [14]:
# Movies directed by Robert Rodriguez
rodriguez_movies = (netflix["director"] == "Robert Rodriguez") & (netflix["type"] == "Movie")
netflix[rodriguez_movies]

,title,director,date_added,type
1384,Spy Kids: All the Time in the World,Robert Rodriguez,2019-02-19,Movie
1416,Spy Kids 3: Game Over,Robert Rodriguez,2019-04-01,Movie
1460,Spy Kids 2: The Island of Lost Dreams,Robert Rodriguez,2019-03-08,Movie
2890,Sin City,Robert Rodriguez,2019-10-01,Movie
3836,Shorts,Robert Rodriguez,2019-07-01,Movie
3883,Spy Kids,Robert Rodriguez,2019-04-01,Movie


In [15]:
# Titles added in May 2019 using .between() on datetime objects
may_2019_titles = netflix["date_added"].between("2019-05-01", "2019-05-31")
netflix[may_2019_titles].head()

,title,director,date_added,type
29,Chopsticks,Sachin Yardi,2019-05-31,Movie
60,Away From Home,NaN,2019-05-08,TV Show
108,Jailbirds,NaN,2019-05-10,TV Show
124,Pegasus,Han Han,2019-05-31,Movie
154,When They See Us,NaN,2019-05-31,TV Show


## 8. Boolean Filtering & Deduplication Cheat Sheet

| Task | Idiomatic Syntax | Key Trap / Consideration |
| :--- | :--- | :--- |
| **Bitwise AND** | `(cond1) & (cond2)` | Must use parentheses around each condition |
| **Bitwise OR** | `(cond1) | (cond2)` | Higher precedence than comparison operators |
| **Bitwise NOT** | `~cond` | Inverts boolean mask cleanly |
| **Multiple Options** | `s.isin([...])` | Preferred over chained `|` clauses |
| **Interval Matching**| `s.between(start, end)` | Works on numbers, strings, and datetimes |
| **Null Detection** | `s.isna()` / `s.notna()` | Never use `s == np.nan` |
| **Nullable Bool** | `s.astype("boolean")` | Prevents `np.nan` from becoming `True` |
| **Drop Duplicates** | `df.drop_duplicates(subset=...)` | Set `keep=False` to drop all duplicate occurrences |

---
## 🎯 9. Technical Interview Corner: Tricky Questions & Drills

### Q1: The Inversion Operator `~`
**Question**: An interviewer asks: *"You have a complex boolean condition `mask = (df['A'] > 10) & (df['B'] == 'X')`. How do you select all rows that do NOT satisfy this condition without rewriting all inequality operators?"*

**Answer**:
Use the tilde **`~`** inversion operator: `df[~mask]`.
By De Morgan's Laws:
`~((A > 10) & (B == 'X'))` $\Longleftrightarrow$ `(A <= 10) | (B != 'X')`.
The `~` operator flips every boolean value in the Series vector efficiently at the hardware bitwise level.

In [16]:
# Demonstration of ~ inversion operator
mask = (employees["Team"] == "Finance") & (employees["Salary"] > 100_000)
print("Employees matching mask:     ", len(employees[mask]))
print("Employees matching ~mask:    ", len(employees[~mask]))
print("Sum of both (matches total): ", len(employees[mask]) + len(employees[~mask]) == len(employees))

Employees matching mask:      40
Employees matching ~mask:     961
Sum of both (matches total):  True


### Q2: How `keep=False` Solves Identity Verification Questions
**Question**: In financial transaction monitoring, how do you isolate all accounts that have duplicate suspect transactions vs accounts with strictly unique transactions?

**Answer**:
- `df[df['account_id'].duplicated(keep=False)]` filters for **all rows that belong to a duplicate set** (both original and subsequent occurrences).
- `df[~df['account_id'].duplicated(keep=False)]` isolates **strictly unique accounts** that only ever appeared once.

In [17]:
# Drill: Separating duplicates vs singleton transactions
tx_demo = pd.DataFrame({
    "Account": ["Acc1", "Acc2", "Acc3", "Acc1", "Acc4", "Acc2"],
    "Amount": [100, 200, 300, 150, 400, 250]
})

print("Accounts appearing multiple times (keep=False):")
display(tx_demo[tx_demo["Account"].duplicated(keep=False)])

print("Accounts appearing STRICTLY ONCE (~keep=False):")
display(tx_demo[~tx_demo["Account"].duplicated(keep=False)])

Accounts appearing multiple times (keep=False):


,Account,Amount
0,Acc1,100
1,Acc2,200
3,Acc1,150
5,Acc2,250


Accounts appearing STRICTLY ONCE (~keep=False):


,Account,Amount
2,Acc3,300
4,Acc4,400


### Q3: Advanced Interview Challenge: Above-Average Earners per Team
**Challenge**: In a single chained expression, filter the `employees` DataFrame to find all employees whose `Salary` is **greater than the average salary of their respective team**!

In [18]:
# Solution using transform to broadcast group mean back to individual rows
above_avg_earners = (
    employees.dropna(subset=["Team", "Salary"])
    .assign(team_avg_salary=lambda df: df.groupby("Team")["Salary"].transform("mean"))
    .query("Salary > team_avg_salary")
    .sort_values(by=["Team", "Salary"], ascending=[True, False])
)

above_avg_earners[["First Name", "Team", "Salary", "team_avg_salary"]].head(10)

,First Name,Team,Salary,team_avg_salary
721,Harold,Business Dev,147417,91866.316832
368,Marilyn,Business Dev,147183,91866.316832
536,Clarence,Business Dev,146589,91866.316832
370,Linda,Business Dev,144001,91866.316832
36,Rachel,Business Dev,142032,91866.316832
933,Doris,Business Dev,141439,91866.316832
419,Dorothy,Business Dev,140136,91866.316832
9,Frances,Business Dev,139852,91866.316832
895,Janice,Business Dev,139791,91866.316832
477,Albert,Business Dev,137840,91866.316832
